In [90]:
from pathlib import Path
import re
from zipfile import ZipFile

import geopandas as gpd
from shapely import box

RGI_DATA_PATH = Path('/home/apbarret/Data/RGI/v07')
RGI_OUTLINE_PATH = RGI_DATA_PATH / 'RGI2000-v7.0-G'

## Create shapefile containing bounding boxes for RGI regions

These are save as a zipfile in `./data/rgi_regions.zip`

In [91]:
# # Uncomment to create shapefiles
# p = re.compile(r'-(\d{2})_(\w*).zip')

# region_id = []
# region_name = []
# region_file = []
# geometry = []
# for fp in RGI_OUTLINE_PATH.glob('*'):
#     m = p.search(fp.name)
#     region_id.append(int(m.groups()[0]))
#     region_name.append(' '.join(m.groups()[1].split('_')))
#     region_file.append(fp.name)
#     gdf = gpd.read_file(fp)
#     geometry.append(box(*gdf.total_bounds))
    
# data = {
#     'id': region_id,
#     'name': region_name,
#     'filename': region_file,
# }

# rgi_region_gdf = gpd.GeoDataFrame(data, geometry=geometry, crs=4326)

# rgi_region_path = RGI_DATA_PATH / "rgi_regions.shp"
# rgi_region_gdf.to_file(rgi_region_path)

In [111]:
region_gdf = gpd.read_file('zip:///home/apbarret/src/glims-ingest-tools/data/rgi_regions.zip')
region_gdf

,id,name,filename,geometry
0,12,caucasus middle east,RGI2000-v7.0-G-12_caucasus_middle_east.zip,"POLYGON ((52.12418 30.89954, 35.46153 30.89954..."
1,17,southern andes,RGI2000-v7.0-G-17_southern_andes.zip,"POLYGON ((-66.06488 -55.45353, -75.23441 -55.4..."
2,19,subantarctic antarctic islands,RGI2000-v7.0-G-19_subantarctic_antarctic_islan...,"POLYGON ((171.20527 -78.30833, -154.57632 -78...."
3,9,russian arctic,RGI2000-v7.0-G-09_russian_arctic.zip,"POLYGON ((105.18141 72.92559, 36.6378 72.92559..."
4,13,central asia,RGI2000-v7.0-G-13_central_asia.zip,"POLYGON ((103.89404 28.13753, 67.42513 28.1375..."
5,16,low latitudes,RGI2000-v7.0-G-16_low_latitudes.zip,"POLYGON ((137.1898 -24.96553, -98.64577 -24.96..."
6,4,arctic canada south,RGI2000-v7.0-G-04_arctic_canada_south.zip,"POLYGON ((-61.50226 58.58964, -86.58278 58.589..."
7,8,scandinavia,RGI2000-v7.0-G-08_scandinavia.zip,"POLYGON ((23.31288 59.5251, 5.3242 59.5251, 5...."
8,3,arctic canada north,RGI2000-v7.0-G-03_arctic_canada_north.zip,"POLYGON ((-64.07569 74.46754, -115.36726 74.46..."
9,18,new zealand,RGI2000-v7.0-G-18_new_zealand.zip,"POLYGON ((175.58764 -45.33325, 167.26978 -45.3..."


In [113]:
region_gdf.explore(column='name')

### Get filename for RGI region file using bbox

In [45]:
bbox = box(137.18071136,  -4.083933  , 137.18715564,  -4.06288835)
rgi_region_gdf[rgi_region_gdf.intersects(bbox)].to_dict()

{'region_id': {5: 16},
 'region_name': {5: 'low latitudes'},
 'region_file': {5: 'RGI2000-v7.0-G-16_low_latitudes.zip'},
 'geometry': {5: <POLYGON ((137.19 -24.966, 137.19 19.181, -98.646 19.181, -98.646 -24.966, 1...>}}

In [118]:
def load_rgi_regions():
    """Returns a GeoDataFrame containing bounding boxes for RGI regions"""
    filepath = 'zip:///home/apbarret/src/glims-ingest-tools/data/rgi_regions.zip'
    return gpd.read_file(filepath)


def get_rgi_filepath(bbox):
    """Returns filepath for RGI file"""
    gdf = load_rgi_regions()
    filename = gdf.loc[gdf.intersects(bbox), 'filename'].array[0]
    return RGI_OUTLINE_PATH / filename

In [122]:
filename = get_rgi_filepath(bbox)
filename

PosixPath('/home/apbarret/Data/RGI/v07/RGI2000-v7.0-G/RGI2000-v7.0-G-16_low_latitudes.zip')

In [121]:
rgi_gdf = gpd.read_file(filename)

In [83]:
minx, miny, maxx, maxy = bbox.bounds
rgi_gdf.explore(
    min_lat=miny,
    min_lon=minx,
    max_lat=maxy,
    max_lon=maxx,
    max_bounds=True,
    zoom_start=12,
    tiles="EsriWorldImagery"
)